# Session 11: Advanced Retrieval with LangChain

## Learning Objectives:

- Understand and implement multiple retrieval strategies for RAG
- Compare naive, BM25, multi-query, parent-document, contextual compression, ensemble, and semantic chunking approaches
- Build RAG chains over a health and wellness knowledge base using LangChain and QDrant

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

---

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

> NOTE: Create a `.env` file in this directory with `OPENAI_API_KEY` and `COHERE_API_KEY` to avoid being prompted each time.

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

In [3]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

In [4]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - Assignment 11 - {uuid4().hex[0:8]}"

## Task 2: Data Collection and Preparation

We'll be using our Health and Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, stress management, habits, and common health concerns.

### Data Preparation

We'll load the wellness guide as a single document, then split it into smaller chunks using a `RecursiveCharacterTextSplitter` for our vector store. We also keep the raw (unsplit) document for use with the Parent Document Retriever and Semantic Chunker later.

In [5]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/HealthWellnessGuide.txt")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
wellness_docs = text_splitter.split_documents(raw_docs)

Let's verify our data was loaded and split correctly!

In [6]:
print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(wellness_docs)}")
print(f"\nExample chunk:\n{wellness_docs[0]}")

Raw documents: 1
Split chunks: 45

Example chunk:
page_content='The Personal Wellness Guide
A Comprehensive Resource for Health and Well-being

PART 1: EXERCISE AND MOVEMENT

Chapter 1: Understanding Exercise Basics

Exercise is one of the most important things you can do for your health. Regular physical activity can improve your brain health, help manage weight, reduce the risk of disease, strengthen bones and muscles, and improve your ability to do everyday activities.' metadata={'source': 'data/HealthWellnessGuide.txt'}


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "wellness_guide".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [7]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    wellness_docs,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide",
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [8]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [9]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [10]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [11]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [12]:
naive_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, then alternate between arching your back up (cat) and letting it sag down (cow). Perform 10-15 repetitions.\n\n- **Bird Dog:** From hands and knees, extend the opposite arm and leg while keeping your core engaged. Hold each extension for 5 seconds, then switch sides. Do 10 repetitions per side.\n\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your lower back against the floor by tightening your abdominal muscles and tilting your pelvis slightly upward. Hold for 10 seconds and repeat 8-12 times.\n\nThese exercises are recommended for alleviating lower back discomfort and can help prevent future episodes when done regularly.'

In [13]:
naive_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical and mental well-being. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep (7-9 hours for adults) is essential for strengthening the immune system, maintaining cognitive function, and promoting emotional stability. Poor sleep or sleep disorders like insomnia can negatively impact health, increasing the risk of various health issues. Therefore, practicing good sleep hygiene, such as maintaining a consistent sleep schedule and creating a relaxing environment, is important for overall health.'

In [14]:
naive_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking water and staying hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gently massaging the temples and neck\n- Using essential oils like peppermint or lavender\n- Maintaining a regular sleep schedule\n- Practicing deep breathing exercises\n- Engaging in progressive muscle relaxation\n- Taking short walks, preferably in nature\n- Listening to calming music\n\nThese approaches can help alleviate stress and headache symptoms naturally.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [15]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(wellness_docs)

We'll construct the same chain - only changing the retriever.

In [16]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [17]:
bm25_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees. Alternately arch your back up (cat) and let it sag down (cow), performing 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while maintaining core stability. Hold each extension for 5 seconds, then switch sides. Aim for 10 repetitions per side.\n- **Pelvic Tilts:** Lie on your back with knees bent. Flatten your lower back against the floor by tightening your abdominal muscles and tilting your pelvis slightly upward. Hold for 10 seconds and repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises can help alleviate lower back discomfort and may prevent future issues.'

In [18]:
bm25_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. Maintaining a regular sleep schedule and creating a comfortable sleep environment—such as keeping the room cool, dark, and quiet—are important for promoting quality sleep. Good sleep hygiene practices, like establishing a relaxing bedtime routine and limiting screen time before bed, help ensure restorative rest. Adequate, restful sleep supports various aspects of health, including immune function, mental well-being, and physical recovery. Conversely, poor sleep or insomnia can negatively affect overall health, so cultivating healthy sleep habits is essential for well-being.'

In [19]:
bm25_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include relaxation techniques such as progressive muscle relaxation, meditation, and deep breathing exercises. Herbal teas like chamomile or valerian root may help promote relaxation and reduce headache symptoms. Additionally, maintaining proper hydration, managing stress through activities like mindfulness or gentle exercise, and ensuring adequate sleep can help alleviate stress-related headaches. However, it is always best to consult with a healthcare provider before starting any new remedies.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer:

example query:  
    "How do I do the Bird Dog exercise?"  

justification:  
    bm25 is based on bag of words, a sparse representation that compares text by the words they both contain  
    embedding based retrieval uses dense vectors and cosine similarity, which captures semantic (meaning) similarity  
    the health and wellness guide uses the exact phrase "Bird Dog" in the lower back pain section  
    bm25 will rank the chunk containing "Bird Dog" highly because the query and document share that exact term  
    embeddings may instead retrieve other chunks about "exercises" or "lower back" that are semantically similar but do not contain the specific exercise name  
    for queries that rely on exact or rare terms (proper names, specific techniques, or phrases that appear verbatim in the corpus), bm25's lexical matching is better than embeddings' semantic matching  


## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [20]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [21]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help alleviate lower back pain include the Cat-Cow Stretch, Bird Dog, and Pelvic Tilts. The Cat-Cow Stretch involves arching and sagging your back while on hands and knees, and should be done for 10-15 repetitions. The Bird Dog requires extending opposite arm and leg from a hands-and-knees position, holding for 5 seconds, and doing 10 repetitions per side. Pelvic Tilts involve lying on your back with knees bent, flattening your back against the floor by tightening your abs and tilting your pelvis, holding for 10 seconds and repeating 8-12 times. Remember to perform these exercises gently and consult with a healthcare professional if you have any concerns.'

In [23]:
contextual_compression_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. It is essential for physical repair, including tissue regeneration, and for mental well-being. During sleep, the body releases hormones that regulate growth and appetite. Adequate sleep, typically 7-9 hours for adults, supports cognitive functions such as memory and learning by allowing the brain to consolidate memories during REM sleep. Poor sleep or sleep disturbances like insomnia can negatively affect health, indicating the importance of creating a comfortable sleep environment to promote restful sleep.'

In [24]:
contextual_compression_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include drinking water to stay hydrated, applying cold or warm compresses to the head or neck, resting in a dark, quiet room, gently massaging the temples and neck, using peppermint or lavender essential oils, maintaining a regular sleep schedule, practicing deep breathing and progressive muscle relaxation, and engaging in grounding techniques or short walks in nature.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [25]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [26]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [27]:
multi_query_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Begin on hands and knees, then alternate between arching your back upward (cat) and letting it sag down (cow). Aim for 10-15 repetitions.\n\n- **Bird Dog:** From hands and knees, extend opposite arm and leg simultaneously while engaging your core. Hold each extension for about 5 seconds, then switch sides. Perform 10 repetitions per side.\n\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your lower back against the floor by tightening your abdominal muscles and tilting your pelvis slightly upward. Hold for 10 seconds, and repeat 8-12 times.\n\nThese exercises are gentle and aimed at stretching and strengthening muscles supporting the lower back. Always consult with a healthcare professional before starting new exercises, especially if you have severe or persistent pain.'

In [28]:
multi_query_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. Adequate sleep (typically 7-9 hours per night) is essential for physical repair, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases important hormones that regulate growth and appetite. Good sleep quality supports immune function, reduces stress, and helps maintain a healthy weight. Conversely, poor sleep or sleep disturbances like insomnia can lead to issues such as fatigue, mood problems, decreased immune response, and other health concerns. Therefore, maintaining healthy sleep habits and creating a conducive sleep environment are crucial for overall health and well-being.'

In [29]:
multi_query_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises (e.g., inhaling for 4 counts, holding, and exhaling)\n- Progressive muscle relaxation, tensing and releasing muscle groups\n- Grounding techniques, such as naming things you see, hear, feel, smell, and taste\n- Taking short walks, preferably in nature\n- Listening to calming music\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of the temples and neck\n- Using essential oils like peppermint or lavender\n- Maintaining a regular sleep schedule\n- Managing stress through regular exercise, social support, and mindfulness practices\n\nThese remedies can help alleviate headache symptoms and reduce stress naturally.'

### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### Answer:

the multi query retriever works by:  
    taking the original user query and using an llm to create n new, reformulated queries  
    retrieving documents for each of those queries  
    using all unique retrieved documents as context  

a single phrasing may align well with only some relevant passages (e.g. one chunk might say "lower back pain" while another says "lumbar discomfort")  
by generating multiple reformulations, the retriever runs several different searches that surface documents matching different wordings and angles  
taking the union of the results means more relevant documents are included in the context pool  
recall is the proportion of relevant documents that are actually retrieved, so casting a wider net via multiple queries increases the chance of retrieving more relevant documents and thus improves recall  


## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. We split the full document into large "parent" chunks (e.g. 2000 characters).
2. Each parent chunk is further split into smaller "child" chunks (e.g. 400 characters).
3. The child chunks are stored in a VectorStore, while the parent chunks are stored in an in-memory docstore.
4. When we query our Retriever, we do a similarity search comparing our query vector to the child chunks.
5. Instead of returning the child chunks, we return their associated parent chunks.

The basic idea is:

- **Search** for small, focused chunks (better semantic matching)
- **Return** big chunks (richer surrounding context)

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by defining our parent and child splitters.

In [30]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [31]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="wellness_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="wellness_parent_child", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [32]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [33]:
parent_document_retriever.add_documents(raw_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [34]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [35]:
parent_document_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'To help with lower back pain, some gentle stretching and strengthening exercises are recommended. These include:\n\n- **Cat-Cow Stretch:** On hands and knees, alternate arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while engaging your core. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over chest, tighten abdominal muscles, and lift your shoulders off the floor. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lying on your back, pull one knee toward your chest, hold for 15-30 seconds, then switch legs.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your lower back against the floor by engaging your abs and tilting your pelvis slightly. Hold for 10 seconds and repeat 8-12 times.\n\nThese exercises can help alleviate lower back discomfort and may prevent future episodes. However,

In [36]:
parent_document_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep significantly impacts overall health in several ways. It is essential for physical recovery, as during sleep, the body repairs tissues and regenerates cells. Sleep also supports mental well-being by consolidating memories and processing information. Additionally, it plays a crucial role in regulating hormones that control growth and appetite. Adults generally need 7-9 hours of sleep per night to maintain optimal health. Poor sleep quality or insufficient sleep can lead to fatigue, low energy, headaches, dry skin, and dizziness, all of which can compromise overall health and well-being. Therefore, practicing good sleep hygiene and ensuring adequate rest are vital for maintaining good health.'

In [37]:
parent_document_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing deep breathing exercises, engaging in progressive muscle relaxation, taking short walks in nature, listening to calming music, and practicing mindfulness or meditation. For headaches specifically, remedies such as staying well-hydrated, applying cold or warm compresses to the head or neck, resting in a dark, quiet room, gentle massage of temples and neck, using peppermint or lavender essential oils, and maintaining a regular sleep schedule can help manage symptoms naturally.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [38]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [39]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [40]:
ensemble_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, then alternate between arching your back up (like a cat) and letting it sag down (like a cow). Repeat this 10-15 times.\n\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold each extension for about 5 seconds, then switch sides. Do this 10 times per side.\n\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis slightly upward. Hold for 10 seconds and repeat 8-12 times.\n\n- **Partial Crunches:** Lie on your back with knees bent, cross your arms over your chest, tighten your stomach muscles, and lift your shoulders off the floor slightly. Hold briefly, then lower back down. Aim for 8-12 repetitions.\n\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat on the ground. Hold for 15-30

In [41]:
ensemble_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep is essential for overall health because it supports physical well-being, mental health, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Getting adequate sleep (7-9 hours per night) in quality sleep cycles helps maintain immune function, mental clarity, and emotional balance. Poor sleep or sleep disturbances like insomnia can negatively impact physical and mental health, increasing the risk of various health issues. Therefore, maintaining good sleep habits and creating a suitable sleep environment are important for overall health and wellness.'

In [42]:
ensemble_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress include practicing deep breathing exercises, progressive muscle relaxation, grounding techniques (such as noticing and naming things around you), taking short walks in nature, and listening to calming music. For headaches, natural remedies include staying well-hydrated by drinking water, applying cold or warm compresses to the head or neck, resting in a dark, quiet room, gently massaging the temples and neck, using essential oils like peppermint or lavender, consuming small amounts of caffeine if appropriate, and maintaining a regular sleep schedule.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [43]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [44]:
semantic_documents = semantic_chunker.split_documents(raw_docs)

Let's create a new vector store.

In [45]:
semantic_vectorstore = QdrantVectorStore.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide_semantic_chunks"
)

We'll use naive retrieval for this example.

In [46]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [47]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [48]:
semantic_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n1. **Cat-Cow Stretch:** Start on hands and knees. Alternate between arching your back up (cat) and letting it sag down (cow). Aim for 10-15 repetitions.\n\n2. **Partial Crunches:** Lie on your back with knees bent, cross arms over chest, tighten your stomach muscles, and raise your shoulders off the floor. Do 8-12 repetitions.\n\n3. **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\n4. **Pelvic Tilts:** Lie on your back with knees bent. Flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises are recommended for alleviating lower back discomfort and preventing future episodes.'

In [49]:
semantic_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep significantly affects overall health in multiple ways. During sleep, your body repairs tissues, which supports physical health and recovery. It also consolidates memories and enhances cognitive functions, crucial for learning and mental clarity. Additionally, sleep regulates hormones related to growth and appetite, helping maintain a healthy weight and energy balance. Consistently getting 7-9 hours of quality sleep boosts immune function, reduces the risk of chronic diseases, improves mood, and enhances overall well-being.'

In [50]:
semantic_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress include deep breathing exercises, progressive muscle relaxation, grounding techniques (such as naming objects you see, hear, feel, smell, and taste), taking short walks in nature, listening to calming music, practicing mindfulness or meditation, and engaging in hobbies or activities you enjoy.\n\nFor headaches, natural remedies include staying well-hydrated by drinking plenty of water, applying cold or warm compresses to the head or neck, resting in a dark and quiet room, gently massaging the temples and neck, using essential oils like peppermint or lavender, and maintaining a regular sleep schedule. Small amounts of caffeine may also help some types of headaches.'

### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### Answer:

behavior:  
    semantic chunking works well on corpora with clean semantic breaks  
    it embeds sentences then combines or splits them based on semantic similarity using thresholding methods (percentile, standard_deviation, interquartile, gradient)  
    with short, highly repetitive text like faqs, many sentences have very similar structure and wording (e.g. "Q: ... A: ..."), so their embeddings and pairwise distances will be similar  
    that leaves little variation in distances, so breakpoints become unclear  
    the algorithm may merge too much (everything into a few large chunks) or break too often (many tiny chunks), and the resulting chunks may not align well with individual q&a pairs  

adjustments:  
    change the thresholding method: try standard_deviation or gradient so that breakpoints are more sensitive to the small variation in distances in a repetitive corpus  
    tune the threshold value: e.g. with percentile, use a lower breakpoint threshold so the algorithm breaks more often when distances are clustered tightly, yielding smaller chunks that are more likely to separate distinct q&a items  
    for heavily faq style content, semantic chunking may be a poor fit because there are few clean semantic breaks; consider a structure based split (e.g. by question–answer pair) in addition to or instead of purely semantic chunking  


---

# 🤝 Breakout Room Part #2

### 🏗️ Activity #1:

Your task is to evaluate the various Retriever methods against each other.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [51]:
# --- 1. Create golden dataset with Ragas synthetic data generation ---
# (Same pattern as Synthetic_Data_Generation_RAGAS & Evaluating_RAG workbooks)
import copy
import time
import numpy as np
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator
from ragas import EvaluationDataset, evaluate, RunConfig
from ragas.metrics import LLMContextRecall, ContextEntityRecall
from langchain_openai import ChatOpenAI

# Wrap notebook's chat_model and embeddings for Ragas (same as Synthetic_Data_Generation / Evaluating_RAG)
generator_llm = LangchainLLMWrapper(chat_model)
generator_embeddings = LangchainEmbeddingsWrapper(embeddings)
generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
# Use raw_docs (full doc); Ragas needs documents >100 tokens (wellness_docs are small chunks)
golden_dataset = generator.generate_with_langchain_docs(raw_docs, testset_size=10)

# --- 2. Evaluate each retriever with retriever-specific Ragas metrics ---
# Use gpt-4.1-mini for evaluator (Evaluating_RAG pattern): nano often fails JSON parsing for Ragas metrics
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
retriever_specific_metrics = [LLMContextRecall(), ContextEntityRecall()]
run_config = RunConfig(timeout=360)

retrievers = [
    ("naive_retriever", naive_retriever),
    ("bm25_retriever", bm25_retriever),
    ("compression_retriever", compression_retriever),
    ("multi_query_retriever", multi_query_retriever),
    ("parent_document_retriever", parent_document_retriever),
    ("ensemble_retriever", ensemble_retriever),
]

results_list = []
for name, retriever in retrievers:
    ds_copy = copy.deepcopy(golden_dataset)
    start = time.perf_counter()
    for test_row in ds_copy:
        docs = retriever.invoke(test_row.eval_sample.user_input)
        test_row.eval_sample.retrieved_contexts = [d.page_content for d in docs]
        test_row.eval_sample.response = ""  # retriever-only eval
        # Cohere Trial key: 10 calls/min. Delay for compression & ensemble (use reranker).
        if name in ("compression_retriever", "ensemble_retriever"):
            time.sleep(6)
    latency_sec = time.perf_counter() - start
    eval_df = ds_copy.to_pandas().replace({np.nan: None})  # NaN -> None for SingleTurnSample
    eval_ds = EvaluationDataset.from_pandas(eval_df)
    ragas_result = evaluate(
        dataset=eval_ds,
        metrics=retriever_specific_metrics,
        llm=evaluator_llm,
        run_config=run_config,
    )
    # EvaluationResult uses __getitem__: result['metric_name'] returns list of per-sample scores
    try:
        cr = float(np.nanmean(ragas_result["context_recall"]))
    except (KeyError, TypeError):
        cr = float("nan")
    try:
        cer = float(np.nanmean(ragas_result["context_entity_recall"]))
    except (KeyError, TypeError):
        cer = float("nan")
    results_list.append({
        "retriever": name,
        "context_recall": cr,
        "context_entity_recall": cer,
        "latency_sec": round(latency_sec, 2),
    })

# --- 3. Compile in a list and paragraph (cost, latency, performance) ---
import pandas as pd
results_df = pd.DataFrame(results_list)
print("Results (retriever-specific metrics, latency):")
print(results_df.to_string(index=False))
print("\n--- Which retriever is best for this Health & Wellness data? ---")
best = results_df.loc[results_df["context_recall"].idxmax(), "retriever"] if results_df["context_recall"].notna().any() else results_df["retriever"].iloc[0]
print(f"**Performance:** {best} achieves the highest context recall on this corpus. "
      "Context recall measures how well retrieved chunks cover the reference answer; "
      "context entity recall measures entity coverage. "
      "**Latency:** Naive and BM25 are fastest (no extra API calls). "
      "Compression (Cohere rerank) and multi-query (LLM reformulations) add latency. "
      "**Cost:** Check LangSmith (LANGCHAIN_TRACING_V2=true) for token/cost breakdown. "
      "Multi-query and compression incur additional LLM/reranker API costs. "
      "For this Health & Wellness Guide, choose based on the cost–latency–performance trade-off.")

/var/folders/g6/fr90m37j7gxbq971zg96s0580000gn/T/ipykernel_83131/1972872295.py:10: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, ContextEntityRecall
/var/folders/g6/fr90m37j7gxbq971zg96s0580000gn/T/ipykernel_83131/1972872295.py:10: DeprecationWarning: Importing ContextEntityRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextEntityRecall
  from ragas.metrics import LLMContextRecall, ContextEntityRecall
/var/folders/g6/fr90m37j7gxbq971zg96s0580000gn/T/ipykernel_83131/1972872295.py:14: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms impo

Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/5 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/22 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/22 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/22 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/22 [00:00<?, ?it/s]

Results (retriever-specific metrics, latency):
                retriever  context_recall  context_entity_recall  latency_sec
          naive_retriever        1.000000               0.372364         2.08
           bm25_retriever        0.530303               0.194159         0.01
    compression_retriever        0.909091               0.326360        76.59
    multi_query_retriever        1.000000               0.320136        20.77
parent_document_retriever        0.969697               0.319923         2.66
       ensemble_retriever        1.000000               0.320933        95.49

--- Which retriever is best for this Health & Wellness data? ---
**Performance:** naive_retriever achieves the highest context recall on this corpus. Context recall measures how well retrieved chunks cover the reference answer; context entity recall measures entity coverage. **Latency:** Naive and BM25 are fastest (no extra API calls). Compression (Cohere rerank) and multi-query (LLM reformulations) add